In [ ]:
"""
danielsinkin97@gmail.com

Interactive visualization for an R^3 -> R^2 -> R^3 linear autoencoder alongside PCA.

- Generates 3D data with intrinsic 2D structure.
- Computes the PCA subspace and reconstructions.
- Trains a linear autoencoder and compares its subspace/reconstructions to PCA.
- Produces an interactive 3D figure (originals, reconstructions, planes) and a 2D latent scatter.
- Saves both figures as standalone HTML and opens them.
"""

from __future__ import annotations

import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import torch
from torch import nn

def make_data(n: int = 800, noise: float = 0.08, seed: int = 7) -> np.ndarray:
    """
    Generate 3D observations with an underlying 2D linear structure plus Gaussian noise and a mean offset.

    Parameters
    ----------
    n : int
        Number of samples to generate.
    noise : float
        Standard deviation of additive Gaussian noise in observation space.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    np.ndarray
        Array of shape (n, 3) with dtype float32 containing the observations.
    """
    rng = np.random.default_rng(seed)
    U = rng.normal(0.0, 1.0, size=(n, 2)).astype(np.float32)
    A = np.array([[0.8, 0.3], [0.2, 0.9], [0.5, -0.1]], dtype=np.float32)
    X = U @ A.T
    X += rng.normal(0.0, noise, size=X.shape).astype(np.float32)
    X += np.array([1.0, -0.7, 0.3], dtype=np.float32)
    return X


def pca_2d(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute the rank-2 PCA subspace and reconstructions.

    Parameters
    ----------
    X : np.ndarray
        Array of shape (n, 3) with dtype float32.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        mu: (3,) mean vector,
        W: (3,2) orthonormal principal directions,
        Z: (n,2) latent coordinates,
        X_hat: (n,3) PCA reconstructions.
    """
    mu = X.mean(axis=0, dtype=np.float32)
    Xc = X - mu
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    V = Vt.T.astype(np.float32)
    W = V[:, :2].astype(np.float32)
    Z = Xc @ W
    X_hat = Z @ W.T + mu
    return mu, W, Z, X_hat


def train_linear_autoencoder(
    X: np.ndarray,
    latent_dim: int = 2,
    epochs: int = 800,
    lr: float = 1e-2,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Train a linear autoencoder (no biases) with MSE loss on mean-centered data.

    Parameters
    ----------
    X : np.ndarray
        Array of shape (n, 3) with dtype float32.
    latent_dim : int
        Latent dimensionality (use 2 to mirror PCA-2).
    epochs : int
        Number of optimization steps.
    lr : float
        Learning rate for Adam.
    seed : int
        Torch random seed.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        mu: (3,) mean vector,
        W_orth: (3,latent_dim) orthonormalized encoder directions spanning the learned subspace,
        Z: (n,latent_dim) latent codes,
        X_hat: (n,3) reconstructions in the original coordinate system.
    """
    torch.manual_seed(seed)
    X_t = torch.tensor(X, dtype=torch.float32)
    mu = X_t.mean(dim=0, keepdim=True)
    Xc = X_t - mu

    class LinAE(nn.Module):
        def __init__(self, d_in: int, d_latent: int) -> None:
            super().__init__()
            self.enc = nn.Linear(d_in, d_latent, bias=False)
            self.dec = nn.Linear(d_latent, d_in, bias=False)

        def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
            z = self.enc(x)
            xhat = self.dec(z)
            return z, xhat

    model = LinAE(3, latent_dim)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        z, xhat = model(Xc)
        loss = ((xhat - Xc) ** 2).mean()
        loss.backward()
        opt.step()

    with torch.no_grad():
        Z, Xc_hat = model(Xc)
        X_hat = (Xc_hat + mu).numpy().astype(np.float32)
        Z = Z.numpy().astype(np.float32)
        W_raw = model.enc.weight.detach().T.numpy()
        Q, _ = np.linalg.qr(W_raw)
        W_orth = Q[:, :latent_dim].astype(np.float32)
        mu_np = mu.squeeze(0).numpy().astype(np.float32)

    return mu_np, W_orth, Z, X_hat


def make_plane(
    mu: np.ndarray, W: np.ndarray, Z: np.ndarray, steps: int = 30
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Build a rectangular grid in latent space and map it to a surface in R^3.

    Parameters
    ----------
    mu : np.ndarray
        Mean vector of shape (3,).
    W : np.ndarray
        Basis matrix of shape (3,2).
    Z : np.ndarray
        Latent coordinates used to set the plotting span, shape (n,2).
    steps : int
        Number of grid steps per axis.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        U: (steps, steps) latent grid for z1,
        V: (steps, steps) latent grid for z2,
        Xs: (steps, steps) surface X coordinates,
        Ys: (steps, steps) surface Y coordinates,
        Zs: (steps, steps) surface Z coordinates.
    """
    span = 2.2 * np.std(Z, axis=0)
    u = np.linspace(-span[0], span[0], steps, dtype=np.float32)
    v = np.linspace(-span[1], span[1], steps, dtype=np.float32)
    U, V = np.meshgrid(u, v)
    grid = np.stack([U.ravel(), V.ravel()], axis=1)
    P = grid @ W.T + mu
    Xs = P[:, 0].reshape(steps, steps)
    Ys = P[:, 1].reshape(steps, steps)
    Zs = P[:, 2].reshape(steps, steps)
    return U, V, Xs, Ys, Zs


def plot_interactive(
    X: np.ndarray,
    mu_pca: np.ndarray,
    W_pca: np.ndarray,
    Z_pca: np.ndarray,
    Xhat_pca: np.ndarray,
    mu_ae: np.ndarray,
    W_ae: np.ndarray,
    Z_ae: np.ndarray,
    Xhat_ae: np.ndarray,
) -> tuple[go.Figure, go.Figure]:
    """
    Create an interactive 3D Plotly figure for originals, reconstructions, and planes,
    and a 2D Plotly figure for latent embeddings (AE aligned to PCA via Procrustes).

    Parameters
    ----------
    X : np.ndarray
        Original observations, shape (n,3).
    mu_pca : np.ndarray
        PCA mean, shape (3,).
    W_pca : np.ndarray
        PCA basis, shape (3,2).
    Z_pca : np.ndarray
        PCA latents, shape (n,2).
    Xhat_pca : np.ndarray
        PCA reconstructions, shape (n,3).
    mu_ae : np.ndarray
        AE mean, shape (3,).
    W_ae : np.ndarray
        AE basis (orthonormalized), shape (3,2).
    Z_ae : np.ndarray
        AE latents, shape (n,2).
    Xhat_ae : np.ndarray
        AE reconstructions, shape (n,3).

    Returns
    -------
    tuple[go.Figure, go.Figure]
        fig3d: interactive 3D figure,
        fig2d: interactive 2D latent scatter.
    """
    Xo, Yo, Zo = X[:, 0], X[:, 1], X[:, 2]
    Xr, Yr, Zr = Xhat_pca[:, 0], Xhat_pca[:, 1], Xhat_pca[:, 2]
    _, _, Xs_p, Ys_p, Zs_p = make_plane(mu_pca, W_pca, Z_pca, steps=40)

    traces = [
        go.Scatter3d(x=Xo, y=Yo, z=Zo, mode="markers", name="Original", marker=dict(size=3, opacity=0.35)),
        go.Scatter3d(x=Xr, y=Yr, z=Zr, mode="markers", name="PCA recon", marker=dict(size=3, opacity=0.85)),
        go.Surface(x=Xs_p, y=Ys_p, z=Zs_p, name="PCA plane", showscale=False, opacity=0.25),
    ]

    Xa, Ya, Za = Xhat_ae[:, 0], Xhat_ae[:, 1], Xhat_ae[:, 2]
    traces.append(
        go.Scatter3d(x=Xa, y=Ya, z=Za, mode="markers", name="Linear AE recon", marker=dict(size=3, opacity=0.85, symbol="diamond"))
    )
    _, _, Xs_a, Ys_a, Zs_a = make_plane(mu_ae, W_ae, Z_ae, steps=40)
    traces.append(go.Surface(x=Xs_a, y=Ys_a, z=Zs_a, name="AE plane", showscale=False, opacity=0.18))

    fig3d = go.Figure(data=traces)
    fig3d.update_layout(
        title="R³ data, reconstructions, and learned 2D subspace (interactive)",
        scene=dict(xaxis_title="x₁", yaxis_title="x₂", zaxis_title="x₃", aspectmode="data"),
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=0, r=0, t=40, b=0),
    )

    z_traces = [
        go.Scatter(x=Z_pca[:, 0], y=Z_pca[:, 1], mode="markers", name="PCA latent z", marker=dict(size=5, opacity=0.85))
    ]

    C = Z_ae.T @ Z_pca
    U_, _, Vt_ = np.linalg.svd(C, full_matrices=False)
    R = U_ @ Vt_
    Z_ae_aligned = Z_ae @ R
    z_traces.append(
        go.Scatter(
            x=Z_ae_aligned[:, 0],
            y=Z_ae_aligned[:, 1],
            mode="markers",
            name="AE latent z (aligned)",
            marker=dict(size=5, opacity=0.7, symbol="diamond"),
        )
    )

    fig2d = go.Figure(data=z_traces)
    fig2d.update_layout(
        title="Latent space (R²)",
        xaxis_title="z₁",
        yaxis_title="z₂",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=0, r=0, t=40, b=0),
    )

    return fig3d, fig2d


def main() -> None:
    """
    Generate data, compute PCA and train the linear autoencoder, plot figures, and save HTML outputs.
    """
    X = make_data(n=1000, noise=0.07, seed=42)
    mu_pca, W_pca, Z_pca, Xhat_pca = pca_2d(X)
    mu_ae, W_ae, Z_ae, Xhat_ae = train_linear_autoencoder(X, latent_dim=2, epochs=1200, lr=5e-3, seed=123)
    fig3d, fig2d = plot_interactive(X, mu_pca, W_pca, Z_pca, Xhat_pca, mu_ae, W_ae, Z_ae, Xhat_ae)

    outdir = Path("figs").joinpath("autoencoder_pca")
    outdir.mkdir(parents=True, exist_ok=True)
    fig3d.write_html(outdir.joinpath("r3_r2_r3_linear_ae_3d.html"), include_plotlyjs="cdn")
    fig2d.write_html(outdir.joinpath("latent_2d.html"), include_plotlyjs="cdn")
    fig3d.show()
    fig2d.show()


if __name__ == "__main__":
    main()